In [1]:
import csv
import os

RESULTS_FILE = os.path.join(os.getcwd(), "results.csv")
COLUMNS = ["section", "item", "metric", "value"]

def save_results(section, rows):
    old_rows = []
    if os.path.exists(RESULTS_FILE):
        with open(RESULTS_FILE, newline="", encoding="utf-8") as f:
            old_rows = [r for r in csv.DictReader(f) if r["section"] != section]

    new_rows = []
    for item, metric, value in rows:
        new_rows.append({"section": section, "item": item, "metric": metric, "value": value})

    with open(RESULTS_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=COLUMNS)
        writer.writeheader()
        writer.writerows(old_rows + new_rows)

In [2]:
# Dữ liệu giả định
synthetic_text = [
    # Medical
    "the doctor treated the patient in the hospital",
    "the physician treated the patient in the hospital",
    "the doctor cured the disease",
    "the physician cured the disease",
    "the patient went to the hospital to cure the disease",
    "the doctor works at the hospital",
    "the physician works at the hospital",
    "the nurse and the doctor helped the patient",
    
    # Animals
    "the cat eats fish",
    "the dog eats meat",
    "the cat chases the mouse",
    "the dog chases the cat",
    
    # Tech, Sports, Fruits
    "the programmer writes code on the computer",
    "i use the computer for work",
    "the players play football on the field",
    "he kicked the football",
    "she ate a banana for breakfast",
    "the monkey likes to eat a banana",

    # car / automobile
    "the car drove on the road", "the automobile drove on the road",
    "he bought a new car", "he bought a new automobile",
    "the driver parked the car in the garage", "the driver parked the automobile in the garage",
    
    # king / queen / man / woman
    "the king is a man", "the queen is a woman",
    "the king rules the kingdom", "the queen rules the kingdom",
    "the king wore a crown", "the queen wore a crown",
    "the man works in the city", "the woman works in the city",
    "the man walked to the market", "the woman walked to the market",
    
    # thủ đô - quốc gia
    "paris is the capital of france", "rome is the capital of italy",
    "berlin is the capital of germany", "madrid is the capital of spain",
    "france is a country in europe", "italy is a country in europe",
    "germany is a country in europe", "spain is a country in europe"
]

# tách từ
base_sentences = [sentence.split() for sentence in synthetic_text]

# Nhân bản dữ liệu
corpus_sentences = base_sentences * 100 

print(f"Tổng số câu trong corpus: {len(corpus_sentences)}")

Tổng số câu trong corpus: 4200


### Experiment 2: Word2Vec

In [3]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=corpus_sentences,
    vector_size=100,    
    window=5,           
    min_count=2,        
    epochs=10,          
    sg=1 #1 cho Skip-gram, 0 cho CBOW
)


print(f"Corpus size (số câu): {model.corpus_count}")
print(f"Vocabulary size: {len(model.wv.key_to_index)}")
print(f"Embedding dimension: {model.vector_size}")
print(f"Window: {model.window}")
print(f"Min count: {model.min_count}")
print(f"Epochs: {model.epochs}")
print(f"Training objective: {'Skip-gram' if model.sg == 1 else 'CBOW'}")

config_rows = [
    ("corpus", "num_sentences", model.corpus_count),
    ("corpus", "vocabulary_size", len(model.wv.key_to_index)),
    ("word2vec", "vector_size", model.vector_size),
    ("word2vec", "window", model.window),
    ("word2vec", "min_count", model.min_count),
    ("word2vec", "epochs", model.epochs),
    ("word2vec", "objective", "Skip-gram" if model.sg == 1 else "CBOW"),
]
save_results("17_word2vec_config", config_rows)

Corpus size (số câu): 4200
Vocabulary size: 80
Embedding dimension: 100
Window: 5
Min count: 2
Epochs: 10
Training objective: Skip-gram


### 18. Inspect embeddings

In [4]:
target_word = "doctor"

if target_word in model.wv.key_to_index:
    print(f"Top 5 từ gần nhất với '{target_word}':")
    top_5 = model.wv.most_similar(target_word, topn=5)
    for word, score in top_5:
        print(f" - {word}: {score:.4f}")

    inspect_rows = []
    for word, score in top_5:
        inspect_rows.append((word, "cosine_similarity", round(float(score), 4)))
    save_results("18_inspect_doctor_top5", inspect_rows)
else:
    print(f"Từ '{target_word}' không có trong từ điển.")

Top 5 từ gần nhất với 'doctor':
 - physician: 0.9112
 - patient: 0.8871
 - at: 0.8796
 - nurse: 0.8681
 - treated: 0.8669


Những từ này gần doctor trong không gian vector không phải  vì trong corpus text, các từ này có ngữ cảnh khá tương đồng.

Ví dụ, cả từ doctor lẫn physician hoặc patient đều thường xuyên xuất hiện xung quanh các động từ như treat, cure, examine hoặc danh từ medicine, disease.

### Experiment 3: Context window

In [5]:
# 3 window size
model_w2 = Word2Vec(sentences=corpus_sentences, vector_size=100, window=2, min_count=2, sg=1)
model_w5 = Word2Vec(sentences=corpus_sentences, vector_size=100, window=5, min_count=2, sg=1)
model_w10 = Word2Vec(sentences=corpus_sentences, vector_size=100, window=10, min_count=2, sg=1)

# So sánh similarity
pairs = [("doctor", "physician"), ("doctor", "hospital"), ("cat", "dog")]

print(f"{'Word pair':<20} | {'Window 2':<10} | {'Window 5':<10} | {'Window 10':<10}")
print("-" * 57)
window_rows = []
for w1, w2 in pairs:
    if w1 in model_w2.wv and w2 in model_w2.wv:
        sim2 = model_w2.wv.similarity(w1, w2)
        sim5 = model_w5.wv.similarity(w1, w2)
        sim10 = model_w10.wv.similarity(w1, w2)
        print(f"{w1+'-'+w2:<20} | {sim2:<10.4f} | {sim5:<10.4f} | {sim10:<10.4f}")
        window_rows.append((f"{w1}-{w2}", "window_2_cosine", round(float(sim2), 4)))
        window_rows.append((f"{w1}-{w2}", "window_5_cosine", round(float(sim5), 4)))
        window_rows.append((f"{w1}-{w2}", "window_10_cosine", round(float(sim10), 4)))

save_results("19_context_window", window_rows)

Word pair            | Window 2   | Window 5   | Window 10 
---------------------------------------------------------
doctor-physician     | 0.9603     | 0.9518     | 0.9658    
doctor-hospital      | 0.9275     | 0.9306     | 0.9511    
cat-dog              | 0.9788     | 0.9835     | 0.9792    


Không phải lúc nào window lớn cũng tốt hơn.

Window nhỏ (vd: 2): Bắt các từ đứng ngay cạnh nhau, có xu hướng phát hiện sự tương đồng về cú pháp và các từ đồng nghĩa. Ví dụ doctor và physician thường đứng thay thế cho nhau ở cùng một vị trí trong câu.

Window lớn (vd: 10): Chú ý tới các từ ở xa, bắt được sự tương đồng về chủ đề. 

### Experiment 4: Embedding dimension

In [6]:
import time

dims = [50, 100, 300]
models_dim = {}
dim_rows = []

for dim in dims:
    start_time = time.time()
    
    m = Word2Vec(sentences=corpus_sentences, vector_size=dim, window=5, min_count=2, sg=1)
    
    train_time = time.time() - start_time
    
    # Dung lượng lưu trữ ma trận Embeddings = Vocabulary size * Dimension * 4 bytes float32
    vocab_size = len(m.wv.key_to_index)
    model_size_mb = (vocab_size * dim * 4) / (1024 * 1024)
    
    print(f"Dimension {dim}:")
    print(f" - Training time: {train_time:.2f} seconds")
    print(f" - Embedding Matrix size: {model_size_mb:.2f} MB\n")
    dim_rows.append((f"dim_{dim}", "training_time_sec", round(train_time, 2)))
    dim_rows.append((f"dim_{dim}", "matrix_size_mb", round(model_size_mb, 2)))
    models_dim[dim] = m

save_results("20_embedding_dimension", dim_rows)

Dimension 50:
 - Training time: 0.12 seconds
 - Embedding Matrix size: 0.02 MB

Dimension 100:
 - Training time: 0.11 seconds
 - Embedding Matrix size: 0.03 MB

Dimension 300:
 - Training time: 0.16 seconds
 - Embedding Matrix size: 0.09 MB



Training time & Model size (Computation): Số chiều càng cao, số lượng trọng số model cần tối ưu càng lớn => thời gian training lâu hơn và dung lượng RAM/ổ cứng tiêu tốn để lưu vector cũng lớn theo.

Capacity (Sức chứa thông tin): Số chiều lớn (100 hoặc 300) cung cấp cho model nhiều không gian biểu diễn hơn. Nó có thể phân biệt nhiều khía cạnh của một từ, phân tách các đa nghĩa => task similarity và analogy chính xác hơn.

Data size: Dimension cao => Corpus phải cực kỳ lớn. Với một corpus nhỏ, nếu set dimension = 300, mô hình sẽ bị overfitting.

### Evaluation: Word similarity

In [7]:
word_pairs = [
    ("doctor", "physician"),
    ("car", "automobile"),
    ("king", "queen"),
    ("cat", "dog"),
    ("computer", "banana")
]

results = []
print(f"{'Word Pair':<25} | {'Similarity Score'}")
print("-" * 45)

for w1, w2 in word_pairs:
    if w1 in model.wv and w2 in model.wv:
        score = model.wv.similarity(w1, w2)
        results.append((w1, w2, score))
    else:
        print(f"{w1}-{w2:<25} | [Từ không có trong Vocab]")

# similarity giảm dần
results.sort(key=lambda x: x[2], reverse=True)

for w1, w2, score in results:
    print(f"{w1 + ' - ' + w2:<25} | {score:.4f}")

similarity_rows = []
rank = 1
for w1, w2, score in results:
    similarity_rows.append((f"{w1}-{w2}", "cosine_similarity", round(float(score), 4)))
    similarity_rows.append((f"{w1}-{w2}", "rank", rank))
    rank += 1
save_results("21_word_similarity", similarity_rows)

Word Pair                 | Similarity Score
---------------------------------------------
car - automobile          | 0.9966
king - queen              | 0.9902
cat - dog                 | 0.9497
doctor - physician        | 0.9112
computer - banana         | 0.4648


Các cặp đồng nghĩa hoàn toàn (doctor - physician, car - automobile) đứng trên cùng với điểm cao nhất.

Các cặp cùng loại (cat - dog, king - queen) có điểm ở mức khá cao (vì chúng cùng là động vật/thú cưng, hoặc cùng là hoàng gia nên thường xuất hiện trong ngữ cảnh câu tương tự nhau).

Cặp không liên quan (computer - banana) thấp nhất nhưng vẫn ở mức 0.4648. => có thể mô hình bắt sự tương đồng về mặt từ loại ngữ pháp thay vì ngữ nghĩa do mô hình thiếu dữ liệu/ corpus quá nhỏ.

### Evaluation: Word analogy

In [8]:
if all(word in model.wv for word in ["king", "woman", "man"]):
    analogy_result = model.wv.most_similar(positive=["king", "woman"], negative=["man"], topn=3)
    
    print("KING - MAN + WOMAN = ?")
    for word, score in analogy_result:
        print(f" - {word}: {score:.4f}")

    analogy_rows = [("king - man + woman", "top1_prediction", analogy_result[0][0])]
    for word, score in analogy_result:
        analogy_rows.append((word, "analogy_score", round(float(score), 4)))
    save_results("22_word_analogy", analogy_rows)
else:
    print("Thiếu từ trong vocabulary.")

KING - MAN + WOMAN = ?
 - queen: 0.9843
 - crown: 0.9302
 - wore: 0.8989


Điều này xảy ra vì Word2Vec học được các mối quan hệ ngữ nghĩa và ngữ cảnh từ dữ liệu. Hướng từ man => woman có thể tương tự hướng từ king => queen. Vì vậy:

king − man + woman = queen

Khi thực hiện phép tính vector, kết quả sẽ nằm gần vector của queen trong không gian embedding.

### 24. Application: Semantic Search

In [9]:
import numpy as np

documents = [
    "the doctor helped the patient",
    "the nurse works at the hospital",
    "the physician cured the disease",
    "the dog chases the cat",
    "the cat eats fish",
    "the programmer writes code on the computer",
    "the king rules the kingdom",
    "the queen wore a crown",
    "he bought a new automobile",
    "she ate a banana",
]

stop_words = ["the", "a", "in", "to", "of", "on", "at", "for", "and", "is", "he", "she", "i"]

def get_words(text):
    # Lấy các từ trong vocab và bỏ stop word
    words = []
    for w in text.split():
        if w in model.wv and w not in stop_words:
            words.append(w)
    return words

def sentence_vector(text):
    # Vector của câu = trung bình vector các từ
    words = get_words(text)
    if len(words) == 0:
        return None
    vectors = [model.wv[w] for w in words]
    return np.mean(vectors, axis=0)

def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [10]:
queries = ["physician", "doctor disease", "queen crown", "car", "cat mouse"]

expansion_rows = []
for query in queries:
    print(f"Query: '{query}'")
    for word in get_words(query):
        related = model.wv.most_similar(word, topn=3)
        related_words = [w for w, s in related]
        print(f"  {word} -> {related_words}")
        expansion_rows.append((query, f"related_to_{word}", " ".join(related_words)))
    print()

save_results("24_query_expansion", expansion_rows)

Query: 'physician'
  physician -> ['at', 'hospital', 'doctor']

Query: 'doctor disease'
  doctor -> ['physician', 'patient', 'at']
  disease -> ['went', 'cure', 'cured']

Query: 'queen crown'
  queen -> ['king', 'crown', 'rules']
  crown -> ['wore', 'king', 'queen']

Query: 'car'
  car -> ['automobile', 'driver', 'garage']

Query: 'cat mouse'
  cat -> ['chases', 'mouse', 'dog']
  mouse -> ['dog', 'chases', 'cat']



In [11]:
def semantic_search(query, top_k=3):
    query_vec = sentence_vector(query)
    query_words = get_words(query)
    results_list = []
    for doc in documents:
        doc_vec = sentence_vector(doc)
        semantic_score = cosine(query_vec, doc_vec)
        # keyword baseline: đếm số từ query xuất hiện nguyên văn trong document
        keyword_score = 0
        for w in query_words:
            if w in doc.split():
                keyword_score += 1
        results_list.append((doc, semantic_score, keyword_score))
    # sắp xếp theo semantic_score giảm dần
    results_list.sort(key=lambda x: x[1], reverse=True)
    return results_list[:top_k]


search_rows = []
for query in queries:
    print(f"Query: '{query}'")
    print(f"  {'Document':<45}{'semantic':>10}{'keyword':>9}")
    top_docs = semantic_search(query, top_k=3)
    rank = 1
    for doc, semantic_score, keyword_score in top_docs:
        print(f"  {doc:<45}{semantic_score:>10.4f}{keyword_score:>9}")
        search_rows.append((query, f"top{rank}_document", doc))
        search_rows.append((query, f"top{rank}_semantic_score", round(float(semantic_score), 4)))
        search_rows.append((query, f"top{rank}_keyword_score", keyword_score))
        rank += 1
    print()

save_results("24_semantic_search", search_rows)

Query: 'physician'
  Document                                       semantic  keyword
  the physician cured the disease                  0.9477        1
  the nurse works at the hospital                  0.8802        0
  the doctor helped the patient                    0.8250        0

Query: 'doctor disease'
  Document                                       semantic  keyword
  the physician cured the disease                  0.9738        1
  the doctor helped the patient                    0.9242        1
  the nurse works at the hospital                  0.9062        0

Query: 'queen crown'
  Document                                       semantic  keyword
  the queen wore a crown                           0.9901        2
  the king rules the kingdom                       0.9290        0
  she ate a banana                                 0.5289        0

Query: 'car'
  Document                                       semantic  keyword
  he bought a new automobile                     